# qwen3.5:9b — Thinking vs Non-Thinking Benchmark (Apple M5 Pro)

Measures the **output** and **latency** of the Steam game recommender's LLM stage with reasoning turned **off** (`think=False`) vs **on** (`think=True`), running fully on the Apple M5 Pro GPU (Metal).

- **Model:** `qwen3.5:9b` served by Ollama, called through LiteLLM (`ollama_chat/` provider)
- **Test query:** *"Find me similar games to GTA V, since the release GTA VI is nowhere in sight"*
- **Method:** ChromaDB retrieval + rerank is run **once**; both generations consume the identical prompt. Only the `think` flag differs, so the latency delta is attributable to reasoning alone.
- **Metrics:** wall-clock latency, prompt / completion tokens, and output throughput (tokens/sec).

> **Why `ollama_chat/` here?** The production app (`recommender.py`) uses the `ollama/` provider with `think=False` and regex-strips any `<think>` block. With that provider a `think=True` call returns an *empty* answer (the reasoning consumes the token budget and is discarded). The `ollama_chat/` chat endpoint instead returns the reasoning separately as `reasoning_content`, cleanly split from the final answer — which is what we need to inspect and report both modes.

## 1. Setup

### 1.1 Test environment

In [1]:
import os, platform, subprocess, time, re, logging, warnings

# Local Ollama calls must bypass any macOS system proxy, else httpx routes
# localhost -> proxy -> 503. (Matches recommender.py.)
os.environ.setdefault("NO_PROXY", "localhost,127.0.0.1")
os.environ.setdefault("no_proxy", "localhost,127.0.0.1")

# Quiet the embedding-model load noise (HF Hub progress bars / tqdm widget warning)
# so the notebook reads cleanly in the report.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="IProgress not found")
try:
    from huggingface_hub.utils import logging as _hf_logging
    _hf_logging.set_verbosity_error()
except Exception:
    pass


def _sysctl(key):
    try:
        return subprocess.check_output(["sysctl", "-n", key], text=True).strip()
    except Exception:
        return "n/a"


_mem = _sysctl("hw.memsize")
print("Chip:        ", _sysctl("machdep.cpu.brand_string"))
print("Unified RAM: ", f"{round(int(_mem) / 1024**3)} GB" if _mem.isdigit() else "n/a")
print("CPU cores:   ", _sysctl("hw.ncpu"))
print("macOS:       ", platform.mac_ver()[0])
print("Python:      ", platform.python_version())
try:
    print("Ollama:      ", subprocess.check_output(["ollama", "--version"], text=True).strip())
except Exception as exc:
    print("Ollama:      ", exc)

Chip:         Apple M5 Pro
Unified RAM:  48 GB
CPU cores:    18
macOS:        26.5
Python:       3.13.9
Ollama:       ollama version is 0.24.0


### 1.2 Benchmark configuration

In [2]:
MODEL = "ollama_chat/qwen3.5:9b"   # chat endpoint -> reasoning returned as reasoning_content
QUERY = "Find me similar games to GTA V, since the release GTA VI is nowhere in sight"
NUM_CTX = 8192       # KV-cache window; large enough for the reasoning trace, still 100% GPU-resident on M5 Pro
MAX_TOKENS = 4096    # generation cap (reasoning + answer)
TEMPERATURE = 0.7    # production setting
SEED = 42            # fixes sampling so re-runs are reproducible

print(f"model={MODEL}  num_ctx={NUM_CTX}  max_tokens={MAX_TOKENS}  temperature={TEMPERATURE}  seed={SEED}")

model=ollama_chat/qwen3.5:9b  num_ctx=8192  max_tokens=4096  temperature=0.7  seed=42


## 2. Shared inputs and harness

### 2.1 Shared RAG context

Reuse the production retrieval pipeline (`recommender.py`): embed the query, pull 15 candidates from ChromaDB, rerank to the top 5, and assemble the exact system + user prompt the app would send. Both timed runs reuse this prompt verbatim.

In [3]:
import sys
from pathlib import Path

# recommender.py lives in the data/ directory (this notebook is in data/notebook/).
_here = Path.cwd()
for _cand in (_here, _here.parent, _here.parent.parent):
    if (_cand / "recommender.py").exists():
        sys.path.insert(0, str(_cand))
        break

from recommender import create_search_engine

engine = create_search_engine()
candidates = engine.retrieve_candidates(QUERY)
ranked = engine.rank_candidates(candidates)

system_prompt = engine._load_system_prompt()
context = engine._format_context(ranked)
user_message = f"User query: {QUERY}\n\nRetrieved games:\n{context}"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message},
]

print(f"Indexed games: {engine.indexed_count:,}")
print(f"Retrieved {len(candidates)} candidates -> reranked to top {len(ranked)}:")
for i, c in enumerate(ranked, 1):
    print(f"  {i}. {c['name']}  (score={c['score']})")

Indexed games: 39,175
Retrieved 15 candidates -> reranked to top 5:
  1. Saints Row: The Third  (score=0.8144)
  2. Vidiot Game  (score=0.8079)
  3. Grand Theft Auto IV: The Complete Edition  (score=0.7971)
  4. Vtuber Simulator : Vtuber模擬器  (score=0.788)
  5. Rival Rampage  (score=0.7838)


### 2.2 Generation and timing helper

`run_generation(think)` issues one LiteLLM call and records wall-clock latency, token usage, and throughput. A one-shot warm-up first loads the weights into GPU memory so the timed runs measure steady-state generation rather than cold model load.

In [4]:
import litellm


def run_generation(think):
    t0 = time.perf_counter()
    resp = litellm.completion(
        model=MODEL,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        think=think,
        num_ctx=NUM_CTX,
        seed=SEED,
    )
    latency = time.perf_counter() - t0
    msg = resp.choices[0].message
    usage = resp.usage
    completion_toks = usage.completion_tokens or 0
    return {
        "think": think,
        "latency_s": latency,
        "answer": (msg.content or "").strip(),
        "reasoning": getattr(msg, "reasoning_content", None),
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": completion_toks,
        "total_tokens": usage.total_tokens,
        "tokens_per_s": completion_toks / latency if latency > 0 else 0.0,
    }


# Warm-up: page the 9B weights into the GPU so timing excludes cold model load.
litellm.completion(model=MODEL, messages=[{"role": "user", "content": "ping"}],
                   max_tokens=8, num_ctx=NUM_CTX, think=False)
print("Model warmed up and resident in GPU memory.")

Model warmed up and resident in GPU memory.


## 3. Benchmark runs

### 3.1 Run A — `think=False` (reasoning off)

In [5]:
res_off = run_generation(think=False)
print(f"Latency:           {res_off['latency_s']:.2f} s")
print(f"Prompt tokens:     {res_off['prompt_tokens']}")
print(f"Completion tokens: {res_off['completion_tokens']}")
print(f"Output speed:      {res_off['tokens_per_s']:.1f} tok/s")
print(f"Reasoning trace:   {'yes' if res_off['reasoning'] else 'none'}")
print("\n----- ANSWER (think=False) -----\n")
print(res_off["answer"])

Latency:           11.06 s
Prompt tokens:     657
Completion tokens: 313
Output speed:      28.3 tok/s
Reasoning trace:   none

----- ANSWER (think=False) -----

If you are looking for the closest spiritual successor to the chaotic freedom of GTA V before the next big title arrives, Saints Row: The Third is your best bet. It offers a massive open world filled with outrageous gameplay scenarios and a heavy dose of comedy that rivals the satire found in Rockstar's latest entry. The community has warmed up to this title significantly, boasting an impressive 96% approval rating from over 68,000 reviews, which suggests it holds up well even years after its release.

For those who prefer a more grounded experience that captures the gritty atmosphere of crime and automobile simulation, Grand Theft Auto IV: The Complete Edition remains a solid choice. While it received a slightly mixed reception compared to the newer titles with an 82% approval rating based on over 175,000 reviews, it offers a

### 3.2 Run B — `think=True` (reasoning on)

In [6]:
res_on = run_generation(think=True)
print(f"Latency:           {res_on['latency_s']:.2f} s")
print(f"Prompt tokens:     {res_on['prompt_tokens']}")
print(f"Completion tokens: {res_on['completion_tokens']}")
print(f"Output speed:      {res_on['tokens_per_s']:.1f} tok/s")
_rwords = len(res_on['reasoning'].split()) if res_on['reasoning'] else 0
print(f"Reasoning trace:   {('yes (~%d words)' % _rwords) if res_on['reasoning'] else 'none'}")
print("\n----- REASONING TRACE (think=True) -----\n")
print(res_on["reasoning"] or "(none returned)")
print("\n----- ANSWER (think=True) -----\n")
print(res_on["answer"])

Latency:           74.84 s
Prompt tokens:     655
Completion tokens: 2499
Output speed:      33.4 tok/s
Reasoning trace:   yes (~1347 words)

----- REASONING TRACE (think=True) -----

Thinking Process:

1.  **Analyze the Request:**
    *   **User Query:** Find games similar to GTA V (Grand Theft Auto V).
    *   **Context:** The user mentions GTA VI is not out yet.
    *   **Role:** Veteran gaming journalist (authoritative, conversational, knowledgeable, honest).
    *   **Constraints:** 2-4 short paragraphs. Plain text only (no markdown, no bold, no italics, no asterisks).
    *   **Content Requirements:** Draw on retrieved game data, explain why they match, mention price/approval ratings, be honest about drawbacks, prioritize strongest matches.

2.  **Analyze the Retrieved Data:**
    *   *Saints Row: The Third ($2.49, 96% approval):* Open World, Action, Comedy, Co-op. Very similar vibe (open world, crime, absurdity). High approval.
    *   *Vidiot Game ($0.00, 90% approval):* Casual

### 3.3 GPU residency check

`ollama ps` confirms the model is served at 100% GPU (Metal) at the configured context size — i.e. the latency figures reflect the M5 Pro GPU, not a CPU fallback.

In [7]:
try:
    print(subprocess.check_output(["ollama", "ps"], text=True))
except Exception as exc:
    print("ollama ps unavailable:", exc)

NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3.5:9b    6488c96fa5fa    8.7 GB    100% GPU     8192       4 minutes from now    



## 4. Results

### 4.1 Comparison summary

In [8]:
import pandas as pd

summary = pd.DataFrame([
    {"mode": "think=False",
     "latency_s": round(res_off["latency_s"], 2),
     "completion_tokens": res_off["completion_tokens"],
     "tok/s": round(res_off["tokens_per_s"], 1),
     "answer_chars": len(res_off["answer"]),
     "reasoning_chars": len(res_off["reasoning"] or "")},
    {"mode": "think=True",
     "latency_s": round(res_on["latency_s"], 2),
     "completion_tokens": res_on["completion_tokens"],
     "tok/s": round(res_on["tokens_per_s"], 1),
     "answer_chars": len(res_on["answer"]),
     "reasoning_chars": len(res_on["reasoning"] or "")},
])

_speedup = res_on["latency_s"] / res_off["latency_s"] if res_off["latency_s"] else float("nan")
print(f"Thinking mode took {_speedup:.1f}x longer "
      f"({res_on['latency_s']:.1f}s vs {res_off['latency_s']:.1f}s) and emitted "
      f"{res_on['completion_tokens']} vs {res_off['completion_tokens']} completion tokens "
      f"(incl. ~{len(res_on['reasoning'] or '')} chars of hidden reasoning).\n")
summary

Thinking mode took 6.8x longer (74.8s vs 11.1s) and emitted 2499 vs 313 completion tokens (incl. ~8608 chars of hidden reasoning).



,mode,latency_s,completion_tokens,tok/s,answer_chars,reasoning_chars
0,think=False,11.06,313,28.3,1488,0
1,think=True,74.84,2499,33.4,1569,8608


### 4.2 Appendix — verbatim generations

Consolidated, copy-paste-ready block for the report appendix: both modes with their metrics, the full reasoning trace (thinking mode only), and the final answers exactly as returned by the model.

In [9]:
for label, res in (("THINKING OFF (think=False)", res_off), ("THINKING ON (think=True)", res_on)):
    print("=" * 72)
    print(f"{label}")
    print(f"latency {res['latency_s']:.2f}s | {res['completion_tokens']} completion tokens "
          f"| {res['tokens_per_s']:.1f} tok/s")
    print("=" * 72)
    if res["reasoning"]:
        print("\n[Reasoning trace]\n")
        print(res["reasoning"])
    print("\n[Final answer]\n")
    print(res["answer"])
    print()

THINKING OFF (think=False)
latency 11.06s | 313 completion tokens | 28.3 tok/s

[Final answer]

If you are looking for the closest spiritual successor to the chaotic freedom of GTA V before the next big title arrives, Saints Row: The Third is your best bet. It offers a massive open world filled with outrageous gameplay scenarios and a heavy dose of comedy that rivals the satire found in Rockstar's latest entry. The community has warmed up to this title significantly, boasting an impressive 96% approval rating from over 68,000 reviews, which suggests it holds up well even years after its release.

For those who prefer a more grounded experience that captures the gritty atmosphere of crime and automobile simulation, Grand Theft Auto IV: The Complete Edition remains a solid choice. While it received a slightly mixed reception compared to the newer titles with an 82% approval rating based on over 175,000 reviews, it offers a deep narrative and a realistic driving model that many fans still